# 7-4절 연습 문제 풀이

이 노트북은 7-4절 연습 문제(7-11 ~ 7-15)의 풀이 예시다. 정답이 하나뿐인 문제가 아니므로 다른 구현도 가능하다.

- 본문 예제 코드는 `code_examples/ch07/07-04_example.ipynb`를 참고한다.
- 각 문제마다 **풀이 해설**과 **문제 검토**를 함께 실었다. 문제 검토는 최종 검토 3단계의 기록이다.

In [1]:
import copy
import random
import re
import time

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, Subset, random_split
from torchvision import datasets, transforms

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DOWNLOAD_ROOT = '../../download'
DATA_DIR = '../../data'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'학습 장치: {device}')

학습 장치: cuda


In [2]:
# 오토인코더 학습용 데이터 — 본문 [코드 7-6]과 같이 표준화 없이 0~1 범위를 유지한다
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.view(-1)),
])
mnist_train = datasets.MNIST(root=DOWNLOAD_ROOT, train=True, download=True, transform=transform)
mnist_test = datasets.MNIST(root=DOWNLOAD_ROOT, train=False, download=True, transform=transform)
ae_train, ae_valid = random_split(mnist_train, [50000, 10000],
                                  generator=torch.Generator().manual_seed(SEED))

BATCH_SIZE = 128
ae_train_loader = DataLoader(ae_train, batch_size=BATCH_SIZE, shuffle=True)
ae_valid_loader = DataLoader(ae_valid, batch_size=BATCH_SIZE, shuffle=False)
print(f'오토인코더 학습용: 훈련 {len(ae_train):,} / 검증 {len(ae_valid):,}')
print(f'평가 데이터셋: {len(mnist_test):,}')

오토인코더 학습용: 훈련 50,000 / 검증 10,000
평가 데이터셋: 10,000


In [3]:
class MNISTAutoEncoder(nn.Module):
    def __init__(self, z_size):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(784, 128), nn.ReLU(),
            nn.Linear(128, z_size),
        )
        self.decoder = nn.Sequential(
            nn.Linear(z_size, 128), nn.ReLU(),
            nn.Linear(128, 784), nn.Sigmoid(),
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))

def train_autoencoder(z_size, epochs=80, patience=5, label=''):
    torch.manual_seed(SEED)
    model = MNISTAutoEncoder(z_size).to(device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    best_loss, best_params, best_epoch, counter = float('inf'), None, 0, 0
    for epoch in range(1, epochs + 1):
        model.train()
        for inputs, _ in ae_train_loader:
            inputs = inputs.to(device)
            optimizer.zero_grad()
            loss = criterion(model(inputs), inputs)
            loss.backward()
            optimizer.step()
        model.eval()
        total, size = 0.0, 0
        with torch.no_grad():
            for inputs, _ in ae_valid_loader:
                inputs = inputs.to(device)
                total += criterion(model(inputs), inputs).item() * inputs.size(0)
                size += inputs.size(0)
        valid_loss = total / size
        if valid_loss < best_loss:
            best_loss, best_epoch, counter = valid_loss, epoch, 0
            best_params = copy.deepcopy(model.state_dict())
        else:
            counter += 1
            if counter >= patience:
                break
    model.load_state_dict(best_params)
    print(f'  {label}오토인코더(z={z_size}) 학습 완료: 최적 에포크 {best_epoch}, 검증 손실 {best_loss:.5f}')
    return model

In [4]:
# 본문 [코드 7-8]의 전이 학습 모델 두 가지
class TransferClassifier(nn.Module):
    """사전 학습된 인코더 + 분류기 계층.

    freeze=True 면 인코더 파라미터를 고정하는 특징 추출 방식,
    False 면 인코더까지 함께 학습하는 미세 조정 방식이 된다.
    """

    def __init__(self, encoder, z_size, freeze=True):
        super().__init__()
        self.encoder = copy.deepcopy(encoder)
        if freeze:
            for parameter in self.encoder.parameters():
                parameter.requires_grad = False
        self.classifier = nn.Linear(z_size, 10)

    def forward(self, x):
        return self.classifier(self.encoder(x))

def make_optimizer(model, freeze, base_lr=0.001, fine_tune_lr=0.0001):
    """미세 조정이면 사전 학습 계층에 훨씬 작은 학습률을 준다(본문 [코드 7-9])."""
    if freeze:
        return optim.Adam(model.classifier.parameters(), lr=base_lr)
    return optim.Adam([
        {'params': model.encoder.parameters(), 'lr': fine_tune_lr},
        {'params': model.classifier.parameters(), 'lr': base_lr},
    ])

@torch.no_grad()
def accuracy(model, loader):
    model.eval()
    correct = total = 0
    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)
        correct += (model(inputs).argmax(dim=1) == labels).sum().item()
        total += labels.size(0)
    return correct / total * 100

def train_transfer(encoder, z_size, freeze, loaders, epochs=1000, patience=50, label=''):
    torch.manual_seed(SEED)
    train_loader, valid_loader, test_loader = loaders
    model = TransferClassifier(encoder, z_size, freeze).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = make_optimizer(model, freeze)
    best_loss, best_params, best_epoch, counter = float('inf'), None, 0, 0
    for epoch in range(1, epochs + 1):
        model.train()
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(inputs), labels)
            loss.backward()
            optimizer.step()
        model.eval()
        total, size = 0.0, 0
        with torch.no_grad():
            for inputs, labels in valid_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                total += criterion(model(inputs), labels).item() * inputs.size(0)
                size += inputs.size(0)
        valid_loss = total / size
        if valid_loss < best_loss:
            best_loss, best_epoch, counter = valid_loss, epoch, 0
            best_params = copy.deepcopy(model.state_dict())
        else:
            counter += 1
            if counter >= patience:
                break
    model.load_state_dict(best_params)
    result = {'best_epoch': best_epoch, 'valid_loss': best_loss,
              'train_accuracy': accuracy(model, train_loader),
              'test_accuracy': accuracy(model, test_loader)}
    print(f'  {label}최적 에포크 {result["best_epoch"]:4d} | 검증 손실 {result["valid_loss"]:.4f} '
          f'| 훈련 정확도 {result["train_accuracy"]:.2f}% | 평가 정확도 {result["test_accuracy"]:.2f}%')
    return model, result

In [5]:
# 본문 예제와 같은 전이 학습용 데이터 — 평가 데이터셋에서 100/100/9800으로 나눈다
#   MNIST 훈련 데이터셋은 오토인코더 학습에 이미 사용했으므로 평가 데이터셋을 쓴다
def make_transfer_loaders(train_size, valid_size, batch_size=32, balanced=False, seed=SEED):
    """평가 데이터셋을 훈련/검증/평가로 나눠 전이 학습용 데이터로더를 만든다."""
    if not balanced:
        generator = torch.Generator().manual_seed(seed)
        test_size = len(mnist_test) - train_size - valid_size
        train_set, valid_set, test_set = random_split(
            mnist_test, [train_size, valid_size, test_size], generator=generator)
    else:
        # 클래스별로 같은 수만큼 훈련 샘플을 고른다
        per_class = train_size // 10
        rng = random.Random(seed)
        by_digit = {d: [] for d in range(10)}
        for idx, label in enumerate(mnist_test.targets.tolist()):
            by_digit[label].append(idx)
        train_idx = []
        for digit in range(10):
            rng.shuffle(by_digit[digit])
            train_idx += by_digit[digit][:per_class]
        rest = [i for i in range(len(mnist_test)) if i not in set(train_idx)]
        rng.shuffle(rest)
        valid_idx, test_idx = rest[:valid_size], rest[valid_size:]
        train_set = Subset(mnist_test, train_idx)
        valid_set = Subset(mnist_test, valid_idx)
        test_set = Subset(mnist_test, test_idx)
    return (DataLoader(train_set, batch_size=batch_size, shuffle=True),
            DataLoader(valid_set, batch_size=batch_size, shuffle=False),
            DataLoader(test_set, batch_size=256, shuffle=False),
            train_set)

def class_counts(subset):
    counts = torch.zeros(10, dtype=torch.long)
    for _, label in subset:
        counts[label] += 1
    return counts

base_loaders = make_transfer_loaders(100, 100)
print('본문 예제와 같은 구성(훈련 100 / 검증 100 / 평가 9,800)')
print(f'  훈련 데이터셋의 클래스별 샘플 수: {class_counts(base_loaders[3]).tolist()}')

본문 예제와 같은 구성(훈련 100 / 검증 100 / 평가 9,800)
  훈련 데이터셋의 클래스별 샘플 수: [11, 14, 12, 6, 8, 8, 11, 9, 11, 10]


## 연습 문제 7-11

> 잠재 벡터의 크기를 10으로 늘려 오토인코더와 두 전이 학습 모델을 학습한 후 결과가 어떻게 바뀌는지 확인해 보자.

In [6]:
encoders = {}
for z_size in (2, 10):
    encoders[z_size] = train_autoencoder(z_size).encoder

print()
results_711 = {}
for z_size in (2, 10):
    for freeze, name in [(True, '특징 추출'), (False, '미세 조정')]:
        label = f'z={z_size:<2d} {name} '
        _, result = train_transfer(encoders[z_size], z_size, freeze, base_loaders[:3], label=label)
        results_711[(z_size, name)] = result

  오토인코더(z=2) 학습 완료: 최적 에포크 77, 검증 손실 0.04181


  오토인코더(z=10) 학습 완료: 최적 에포크 77, 검증 손실 0.01599



  z=2  특징 추출 최적 에포크  882 | 검증 손실 1.3788 | 훈련 정확도 52.00% | 평가 정확도 52.13%


  z=2  미세 조정 최적 에포크  322 | 검증 손실 1.3929 | 훈련 정확도 72.00% | 평가 정확도 51.08%


  z=10 특징 추출 최적 에포크  280 | 검증 손실 0.7483 | 훈련 정확도 93.00% | 평가 정확도 81.43%


  z=10 미세 조정 최적 에포크  234 | 검증 손실 0.8202 | 훈련 정확도 98.00% | 평가 정확도 80.54%


In [7]:
print(f'{"잠재 크기":>9} {"방식":>10} {"최적 에포크":>11} {"검증 손실":>10} {"훈련 정확도":>11} {"평가 정확도":>11}')
print('-' * 70)
for (z_size, name), result in results_711.items():
    print(f'{z_size:9d} {name:>10} {result["best_epoch"]:11d} {result["valid_loss"]:10.4f} '
          f'{result["train_accuracy"]:10.2f}% {result["test_accuracy"]:10.2f}%')

print()
print('잠재 벡터 크기를 2에서 10으로 늘렸을 때의 평가 정확도 변화')
for name in ('특징 추출', '미세 조정'):
    before = results_711[(2, name)]['test_accuracy']
    after = results_711[(10, name)]['test_accuracy']
    print(f'  {name}: {before:.2f}% -> {after:.2f}% ({after - before:+.2f}%p)')

    잠재 크기         방식      최적 에포크      검증 손실      훈련 정확도      평가 정확도
----------------------------------------------------------------------
        2      특징 추출         882     1.3788      52.00%      52.13%
        2      미세 조정         322     1.3929      72.00%      51.08%
       10      특징 추출         280     0.7483      93.00%      81.43%
       10      미세 조정         234     0.8202      98.00%      80.54%

잠재 벡터 크기를 2에서 10으로 늘렸을 때의 평가 정확도 변화
  특징 추출: 52.13% -> 81.43% (+29.30%p)
  미세 조정: 51.08% -> 80.54% (+29.46%p)


### 풀이 해설

**무엇을 다시 학습해야 하는가**

지문이 "오토인코더**와** 두 전이 학습 모델을 학습한 후"라고 한 것이 중요하다.
잠재 벡터 크기는 **인코더의 출력 크기**이므로, 오토인코더부터 다시 학습해야 한다.
그리고 분류기 계층의 입력 크기(`nn.Linear(z_size, 10)`)도 함께 바뀐다.

**결과 해석**

잠재 벡터를 2에서 10으로 늘리면 **두 방식 모두 평가 정확도가 크게 오른다.**
본문 p32가 낮은 정확도의 원인으로 지목한 두 가지 중 **첫 번째가 해소되기 때문**이다.

> 첫째, 잠재 벡터의 크기를 2로 매우 작게 잡았다. … 크기 2의 잠재 벡터는 숫자 7과 9처럼 형태가 비슷한
> 숫자들을 구분할 만한 공간을 확보하지 못한다.

연습 문제 7-8에서 확인했듯 잠재 벡터가 커지면 **숫자를 구분하는 정보가 더 많이 담긴다.**
인코더가 이미 잘 나눠 놓은 공간을 분류기가 선형 경계로 가르기만 하면 되므로,
**분류기가 하는 일이 훨씬 쉬워진다.**

여기서 **전이 학습의 성능이 사전 학습된 모델의 품질에 달려 있다**는 사실이 드러난다.
분류기 계층은 그대로인데 인코더가 좋아진 것만으로 정확도가 오른다.
본문 p28이 "대규모 데이터를 잘 학습한 모델은 … 강건함을 가진다. 강건함을 갖춘 모델을 재활용하면
이미 검증된 일반화 능력을 그대로 이어받을 수 있다"고 한 것이 이 뜻이다.

**두 방식의 차이도 함께 보자.** 잠재 벡터가 2일 때는 두 방식의 차이가 크지 않다.
인코더가 담을 수 있는 정보 자체가 부족해 **미세 조정으로 고칠 여지가 별로 없기 때문**이다.
잠재 벡터가 커지면 미세 조정이 손댈 수 있는 공간이 넓어진다.

다만 본문 p32가 지적한 **두 번째 원인(훈련 샘플 100개)은 그대로 남아 있다.**
클래스당 10개꼴이라 어떤 모델도 한계가 있다. 그 원인을 건드리는 것이 다음 문제(7-12)다.

### 문제 검토

- **적절성: 적합. 본문의 해명을 검증하는 문제다.** 본문 p32는 낮은 정확도의 원인으로 '잠재 벡터가 너무 작다'와
  '데이터가 너무 적다' 둘을 든다. **7-11이 첫 번째를, 7-12가 두 번째를 각각 검증**하는 구조라 짜임새가 좋다.
- **[검토] '오토인코더와 두 전이 학습 모델을'이라고 명시한 것이 정확하다.** 인코더를 다시 학습해야 한다는
  것을 빠뜨리면 형태 불일치 오류가 난다. 지문이 이를 챙기고 있다.
- **[검토] 10이라는 값의 선택.** 클래스 수와 같은 10이라 '클래스마다 축 하나씩'이라는 오해를 부를 수 있지만,
  본문 p5가 이미 "임베딩의 축은 사람이 이해하는 언어적 개념과 일대일로 대응되지 않는다"고 잠재 차원을
  설명해 두었으므로 큰 문제는 아니다. 오히려 **'왜 10으로 늘렸는데 100%가 안 나오는가'**를 생각하게 만드는
  좋은 미끼일 수 있다.
- **[검토] '결과가 어떻게 바뀌는지'가 다소 열려 있다.** 정확도만 보고 "좋아졌네"로 끝날 수 있는데,
  이 문제의 소득은 **왜 좋아졌는지**를 본문 p32의 설명과 연결하는 것이다.
  본문이 이미 원인을 밝혀 두었으므로 그것을 확인하라고 명시하면 연결이 분명해진다.

**윤문안**

> **7-11** 잠재 벡터의 크기를 10으로 늘려 오토인코더와 두 전이 학습 모델을 학습한 후 결과가 어떻게 바뀌는지
> 확인해 보자. 그리고 그 변화가 본문에서 설명한 두 가지 극단적인 설정 중 어느 쪽과 관련 있는지 생각해 보자.

## 연습 문제 7-12

> 잠재 벡터의 크기가 2인 원래의 전이 학습 예제에서 훈련, 검증, 평가 데이터셋의 샘플 수를 각각
> 1,000개, 1,000개, 8,000개로 조정하고 모델을 학습한 후 결과가 어떻게 바뀌는지를 확인해 보자.
> 만약 특징 추출 방식과 미세 조정 방식 모델의 정확도 차이가 벌어진다면 그 이유가 무엇인지 설명해 보자.

In [8]:
large_loaders = make_transfer_loaders(1000, 1000)
print(f'훈련 데이터셋의 클래스별 샘플 수: {class_counts(large_loaders[3]).tolist()}')
print()

results_712 = {}
for freeze, name in [(True, '특징 추출'), (False, '미세 조정')]:
    _, result = train_transfer(encoders[2], 2, freeze, large_loaders[:3],
                               label=f'{name}(1000) ')
    results_712[name] = result

훈련 데이터셋의 클래스별 샘플 수: [106, 117, 109, 91, 86, 91, 105, 95, 102, 98]



  특징 추출(1000) 최적 에포크  326 | 검증 손실 1.1788 | 훈련 정확도 58.70% | 평가 정확도 58.15%


  미세 조정(1000) 최적 에포크  144 | 검증 손실 0.6801 | 훈련 정확도 92.00% | 평가 정확도 77.44%


In [9]:
print(f'{"훈련 샘플":>9} {"방식":>10} {"최적 에포크":>11} {"훈련 정확도":>11} {"평가 정확도":>11}')
print('-' * 60)
for name in ('특징 추출', '미세 조정'):
    r = results_711[(2, name)]
    print(f'{100:9d} {name:>10} {r["best_epoch"]:11d} {r["train_accuracy"]:10.2f}% {r["test_accuracy"]:10.2f}%')
for name in ('특징 추출', '미세 조정'):
    r = results_712[name]
    print(f'{1000:9d} {name:>10} {r["best_epoch"]:11d} {r["train_accuracy"]:10.2f}% {r["test_accuracy"]:10.2f}%')

print()
for size, results in [(100, {n: results_711[(2, n)] for n in ('특징 추출', '미세 조정')}),
                      (1000, results_712)]:
    gap = results['미세 조정']['test_accuracy'] - results['특징 추출']['test_accuracy']
    print(f'훈련 샘플 {size:,}개일 때 두 방식의 평가 정확도 차이: {gap:+.2f}%p')

    훈련 샘플         방식      최적 에포크      훈련 정확도      평가 정확도
------------------------------------------------------------
      100      특징 추출         882      52.00%      52.13%
      100      미세 조정         322      72.00%      51.08%
     1000      특징 추출         326      58.70%      58.15%
     1000      미세 조정         144      92.00%      77.44%

훈련 샘플 100개일 때 두 방식의 평가 정확도 차이: -1.05%p
훈련 샘플 1,000개일 때 두 방식의 평가 정확도 차이: +19.29%p


### 풀이 해설

**결과의 두 방향**

훈련 샘플을 100개에서 1,000개로 늘리면 두 방식 모두 정확도가 오른다.
본문 p32가 지목한 **두 번째 원인**이 완화되기 때문이다.

> 둘째, 훈련 데이터셋이 지나치게 작고 불균형이 심하다. 클래스가 10개나 되는 분류 문제에서 총 100개의 샘플은
> 클래스당 평균 10개에 불과하다.

다만 **잠재 벡터 크기가 2라는 첫 번째 원인은 그대로**이므로, 정확도에는 천장이 있다.
크기 2의 잠재 공간에서 7과 9가 겹친다면(본문 그림 7-11 오른쪽) 데이터를 아무리 늘려도 구분할 수 없다.
**데이터가 해결하는 문제와 표현력이 해결하는 문제는 다르다.**

**두 방식의 차이가 벌어지는 이유**

이것이 지문의 두 번째 물음이자 이 문제의 핵심이다. 답은 **두 방식이 학습하는 대상의 크기 차이**에 있다.

| | 특징 추출 | 미세 조정 |
|---|---|---|
| 학습하는 파라미터 | 분류기만 (`z_size × 10 + 10` = **30개**) | 인코더 + 분류기 (**약 10만 개**) |
| 인코더 | 고정 | 함께 최적화 |
| 잠재 공간 | **오토인코더가 만든 그대로** | **분류에 유리하게 재배치** |

**훈련 샘플이 100개일 때**는 미세 조정이 별로 유리하지 않다.
10만 개 파라미터를 100개 샘플로 조정하면 **금세 과적합**되기 때문이다.
본문 p28의 설명 그대로다.

> 데이터가 적을 때는 과적합을 피할 수 있어 **특징 추출 방식이 안전하다.**

**훈련 샘플이 1,000개가 되면** 상황이 달라진다. 미세 조정이 인코더를 **분류 목적에 맞게 다시 배치할**
여유가 생긴다. 오토인코더의 인코더는 '재현을 잘하도록' 학습된 것이지 '분류를 잘하도록' 학습된 것이 아니므로,
**손댈 여지가 실제로 있다.** 특징 추출 방식은 이 여지를 활용할 수 없다. 30개 파라미터로 할 수 있는 일이
정해져 있기 때문이다.

즉 **차이가 벌어지는 이유는 '미세 조정이 갑자기 좋아져서'가 아니라 '특징 추출이 먼저 한계에 닿아서'**다.
특징 추출 방식은 **인코더가 만든 잠재 공간의 품질이 곧 성능의 상한**이다.

본문 p28의 다음 설명이 이 실험으로 확인된다.

> 반대로 데이터가 충분하다면 미세 조정으로 사전 학습된 모델까지 새 데이터에 맞춰 잠재 성능을 끌어올릴 수 있다.

**덧붙임 — 훈련 정확도를 함께 보자.** 미세 조정 쪽의 훈련 정확도가 평가 정확도보다 훨씬 높다면
여전히 과적합이 남아 있다는 뜻이다. 두 방식의 훈련·평가 정확도 차이를 견주면
**'표현력이 늘어난 대가'**도 함께 읽을 수 있다.

### 문제 검토

- **적절성: 적합. 7장에서 가장 잘 설계된 연습 문제 중 하나다.** 조건 하나(데이터 양)만 바꾸고
  나머지는 고정해 **변수를 통제한 실험**이 된다. 그리고 두 번째 물음이 관찰을 설명으로 밀어 올린다.
- **★ [검토] "만약 … 차이가 벌어진다면"이라는 조건절이 좋다.** 결과를 단정하지 않으므로
  실행 환경에 따라 차이가 작게 나와도 문제가 틀리지 않는다. 연습 문제 6-8의 세 번째 물음과 같은 방식이다.
  다만 **차이가 벌어지지 않는 경우에는 답할 것이 없어진다.** "벌어지지 않는다면 그 이유도 생각해 보자"를
  덧붙이면 어느 쪽 결과든 생각할 거리가 남는다.
- **[검토] 1,000 / 1,000 / 8,000이라는 수치가 적절하다.** 평가 데이터셋 1만 개를 남김없이 쓰고,
  훈련 샘플이 열 배가 되어 변화가 뚜렷하다. 검증 샘플도 함께 늘려 조기 종료가 안정적으로 동작한다.
- **[검토] 잠재 벡터 크기를 2로 고정한 것이 의도적이고 정확하다.** 7-11에서 이미 크기를 바꿔 봤으므로,
  여기서는 **다른 변수를 건드리지 않아야** 데이터 양의 효과만 분리해 볼 수 있다.
  "원래의 전이 학습 예제에서"라는 표현이 이를 분명히 한다.

**윤문안**

> **7-12** 잠재 벡터의 크기가 2인 원래의 전이 학습 예제에서 훈련, 검증, 평가 데이터셋의 샘플 수를 각각
> 1,000개, 1,000개, 8,000개로 조정하고 모델을 학습한 후 결과가 어떻게 바뀌는지를 확인해 보자.
> 특징 추출 방식과 미세 조정 방식 모델의 정확도 차이가 벌어진다면 그 이유가 무엇인지,
> 벌어지지 않는다면 무엇이 두 방식의 성능을 함께 묶어 두고 있는지 설명해 보자.

## 연습 문제 7-13 [도전 문제]

> [연습 문제 7-12]에서 데이터셋 구성을 바꿀 때 전이 학습용 훈련 데이터셋이 클래스별 100개씩 균일한
> 1,000개의 샘플로 구성되도록 데이터셋을 분리해 보자.

※ 본문 지문은 **[연습 문제 7-13]**이라고 되어 있으나 자기 자신을 가리키므로 **[연습 문제 7-12]**가 맞다(2단계 보고서 3번 항목).

In [10]:
balanced_loaders = make_transfer_loaders(1000, 1000, balanced=True)
print('무작위 분리 (7-12)')
print(f'  클래스별 샘플 수: {class_counts(large_loaders[3]).tolist()}')
print(f'  최소 {class_counts(large_loaders[3]).min().item()}개, 최대 {class_counts(large_loaders[3]).max().item()}개')
print('클래스 균형 분리 (7-13)')
print(f'  클래스별 샘플 수: {class_counts(balanced_loaders[3]).tolist()}')
print(f'  최소 {class_counts(balanced_loaders[3]).min().item()}개, 최대 {class_counts(balanced_loaders[3]).max().item()}개')
print()

results_713 = {}
for freeze, name in [(True, '특징 추출'), (False, '미세 조정')]:
    _, result = train_transfer(encoders[2], 2, freeze, balanced_loaders[:3],
                               label=f'{name}(균형) ')
    results_713[name] = result

무작위 분리 (7-12)
  클래스별 샘플 수: [106, 117, 109, 91, 86, 91, 105, 95, 102, 98]


  최소 86개, 최대 117개
클래스 균형 분리 (7-13)
  클래스별 샘플 수: [100, 100, 100, 100, 100, 100, 100, 100, 100, 100]


  최소 100개, 최대 100개



  특징 추출(균형) 최적 에포크  348 | 검증 손실 1.3142 | 훈련 정확도 60.20% | 평가 정확도 59.60%


  미세 조정(균형) 최적 에포크  133 | 검증 손실 0.8266 | 훈련 정확도 88.80% | 평가 정확도 76.85%


In [11]:
print(f'{"분리 방법":>12} {"방식":>10} {"훈련 정확도":>11} {"평가 정확도":>11}')
print('-' * 50)
for label, results in [('무작위', results_712), ('클래스 균형', results_713)]:
    for name in ('특징 추출', '미세 조정'):
        r = results[name]
        print(f'{label:>12} {name:>10} {r["train_accuracy"]:10.2f}% {r["test_accuracy"]:10.2f}%')

print()
for name in ('특징 추출', '미세 조정'):
    diff = results_713[name]['test_accuracy'] - results_712[name]['test_accuracy']
    print(f'{name}: 균형 분리로 평가 정확도 {diff:+.2f}%p')

       분리 방법         방식      훈련 정확도      평가 정확도
--------------------------------------------------
         무작위      특징 추출      58.70%      58.15%
         무작위      미세 조정      92.00%      77.44%
      클래스 균형      특징 추출      60.20%      59.60%
      클래스 균형      미세 조정      88.80%      76.85%

특징 추출: 균형 분리로 평가 정확도 +1.45%p
미세 조정: 균형 분리로 평가 정확도 -0.59%p


### 풀이 해설

**구현 방법**

`random_split()`은 클래스를 신경 쓰지 않으므로 다른 방법이 필요하다. 세 단계로 하면 된다.

1. **클래스별로 인덱스를 모은다.** `mnist_test.targets`를 훑어 숫자별 인덱스 목록을 만든다.
2. **각 목록을 섞어 앞에서 100개씩 꺼낸다.** 섞지 않으면 늘 같은 샘플이 뽑혀 재현성은 좋지만 편향될 수 있다.
3. **나머지로 검증·평가 데이터셋을 만든다.** `torch.utils.data.Subset`으로 인덱스 목록에서 부분 집합을 만든다.

핵심은 **`Subset`을 쓰는 것**이다. `random_split()`도 내부적으로 `Subset`을 반환하므로,
인덱스를 직접 고른 `Subset`을 만들면 데이터로더 이후의 코드를 그대로 쓸 수 있다.

**왜 이것이 필요한가**

본문 p32가 지적한 문제다.

> 게다가, 데이터셋을 분할할 때 다른 조건 없이 무작위로 분리한 결과 100개의 훈련 샘플 중 숫자 1의 샘플은
> 14개이지만 숫자 3의 샘플은 6개만 포함되어 있다.

위 실행 결과에서 무작위 분리의 클래스별 샘플 수를 보면 편차가 있다.
샘플이 적은 클래스는 **배울 기회가 적어 정확도가 낮아지고**, 많은 클래스는 유리해진다.
게다가 모델이 애매한 입력에서 **더 흔한 클래스로 기우는** 경향도 생긴다.

**결과 해석**

훈련 샘플이 1,000개일 때는 **균형을 맞춰도 개선 폭이 크지 않다.** 이유가 두 가지다.

**(1) 샘플이 많아지면 무작위 분리도 저절로 균형에 가까워진다.** 100개를 뽑을 때는 클래스당 6~14개로
두 배 넘게 차이 나지만, 1,000개를 뽑으면 클래스당 90~110개 정도로 편차의 **비율**이 훨씬 작아진다.
**클래스 불균형은 데이터가 적을수록 심각한 문제다.**

**(2) 잠재 벡터 크기 2라는 더 큰 제약이 남아 있다.** 7과 9가 잠재 공간에서 겹쳐 있다면
두 클래스의 샘플 수를 똑같이 맞춰도 구분되지 않는다.

**그래서 이 문제의 진짜 소득은 '성능이 얼마나 올랐나'가 아니다.**
**데이터를 나누는 방식 자체가 실험 설계의 일부**라는 것을 알게 되는 데 있다.
본문 p32가 클래스 불균형을 낮은 정확도의 원인으로 든 만큼, 그 원인을 **직접 제거해 보고
얼마나 기여했는지 재 보는 것**이 이 문제가 시키는 일이다.

**덧붙임 — 100개로 실험하면 차이가 더 크게 나타난다.** 본문이 지적한 상황(클래스당 6~14개)을
그대로 재현하려면 훈련 샘플 100개에서 균형 분리를 해 보는 편이 낫다.
**클래스당 10개씩 정확히 맞춘 100개**와 무작위 100개를 견주면 불균형의 영향이 또렷해진다.

### 문제 검토

- **★★ [검토] 참조 번호가 자기 자신을 가리킨다.** "**[연습 문제 7-13]**에서 데이터셋 구성을 바꿀 때"는
  **[연습 문제 7-12]**여야 한다. 2단계 보고서 3번 항목 참조.
- **적절성: 도전 문제로 적합하다.** 본문 p32가 클래스 불균형을 원인으로 지목만 하고 넘어가는데,
  이 문제가 그것을 **직접 제거해 보게** 한다. `random_split()`만 써 온 독자에게
  **데이터를 원하는 기준으로 나누는 법**을 익히게 하는 실용적인 연습이기도 하다.
- **★ [검토] 무엇을 확인하라는 지시가 없다.** 지문은 "데이터셋을 분리해 보자"로 끝난다.
  나누기만 하고 학습하지 않으면 **불균형이 실제로 얼마나 영향을 주는지 알 수 없다.**
  7장의 다른 문제들이 모두 '학습한 후 결과를 확인해 보자'로 끝나는 것과도 형식이 다르다.
  → **"모델을 학습해 7-12의 결과와 비교해 보자"를 덧붙여야 한다.**
- **★ [검토] 1,000개에서는 불균형의 효과가 잘 드러나지 않는다.**
  본문이 문제 삼은 상황은 **훈련 샘플 100개일 때**(클래스당 6~14개)인데,
  이 문제는 1,000개 구성에서 균형을 맞추라고 한다. 1,000개면 무작위로 뽑아도 클래스당 90~110개 정도라
  **편차의 비율이 이미 작다.** 그래서 균형을 맞춰도 개선이 미미하고, 독자는 "별 차이 없네"로 끝낸다.
  → **훈련 샘플 100개(클래스당 10개씩)에서도 함께 비교하게 하면** 본문이 지적한 문제와 정확히 맞물린다.
- **[검토] 도전 문제로 분류한 것이 타당하다.** `random_split()`으로는 풀 수 없고,
  `Subset`과 인덱스 조작을 알아야 하므로 난도가 한 단계 높다.

**윤문안**

> **7-13** [도전 문제] **[연습 문제 7-12]**에서 데이터셋 구성을 바꿀 때 전이 학습용 훈련 데이터셋이
> 클래스별 100개씩 균일한 1,000개의 샘플로 구성되도록 데이터셋을 분리해 보자.
> **그리고 두 전이 학습 모델을 학습해 [연습 문제 7-12]의 결과와 비교해 보자.**
> **클래스별 10개씩 균일한 100개의 훈련 데이터셋으로도 같은 비교를 해 보면, 훈련 샘플 수에 따라
> 클래스 불균형의 영향이 어떻게 달라지는지 확인할 수 있다.**

## 연습 문제 7-14

> 임베딩이나 오토인코더 등 데이터 속에 숨은 의미를 추출할 수 있는 모델만 전이 학습에 사용되지는 않는다.
> 일반적인 분류 모델이 학습한 저수준 특징 추출 능력을 활용하기 위한 전이 학습도 고려해 볼 만한 전략이다.
> 예를 들어 30x30 크기의 이미지에서 개와 고양이를 분류하도록 학습된 `some_model`이 있다고 하자.
> 이 모델을 재활용하면 새로운 작업, 즉 이미지 안에 동물이 있는지 없는지를 판단하는 모델을 만들 수 있다.
> 다음 [코드 7-10]과 같은 구조로 정의되어 개와 고양이를 분류하도록 학습된 `some_model`을 재활용해,
> 이미지에 개 또는 고양이가 있는지 없는지를 판별하는 모델 클래스를 정의해 보자.

In [12]:
# 본문 [코드 7-10]의 some_model
some_model = nn.Sequential(
    nn.Linear(900, 128),
    nn.ReLU(),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Linear(64, 2)
)
print(some_model)
print()
print('계층별 역할')
print('  [0] Linear(900, 128)  ]')
print('  [1] ReLU()            ]-- 특징 추출기 (저수준 -> 고수준)')
print('  [2] Linear(128, 64)   ]')
print('  [3] ReLU()            ]')
print('  [4] Linear(64, 2)     ]-- 분류기 (개 / 고양이)  <- 이 부분은 쓸 수 없다')

Sequential(
  (0): Linear(in_features=900, out_features=128, bias=True)
  (1): ReLU()
  (2): Linear(in_features=128, out_features=64, bias=True)
  (3): ReLU()
  (4): Linear(in_features=64, out_features=2, bias=True)
)

계층별 역할
  [0] Linear(900, 128)  ]
  [1] ReLU()            ]-- 특징 추출기 (저수준 -> 고수준)
  [2] Linear(128, 64)   ]
  [3] ReLU()            ]
  [4] Linear(64, 2)     ]-- 분류기 (개 / 고양이)  <- 이 부분은 쓸 수 없다


In [13]:
class AnimalDetector(nn.Module):
    """some_model의 특징 추출기를 재활용해 동물의 유무를 판별하는 모델.

    마지막 선형 계층(개/고양이 분류기)은 목적이 다르므로 버리고,
    그 앞까지를 특징 추출기로 가져와 새 분류기를 붙인다.

    주의: copy.deepcopy() 없이 children()을 그대로 가져오면 사전 학습 모델과
          '같은 객체'를 공유하게 되어, requires_grad를 바꿀 때 원본까지 함께 바뀐다.
    """

    def __init__(self, pretrained, freeze=True):
        super().__init__()
        # nn.Sequential 은 슬라이싱으로 앞부분만 떼어 올 수 있다
        #   반드시 복사본을 떠야 원본 모델이 오염되지 않는다
        self.features = nn.Sequential(*list(copy.deepcopy(pretrained).children())[:-1])
        if freeze:
            for parameter in self.features.parameters():
                parameter.requires_grad = False
        # 새 분류기: 64차원 특징 벡터 -> 2개 클래스(동물 있음 / 없음)
        self.classifier = nn.Linear(64, 2)

    def forward(self, x):               # (B, 900)
        x = self.features(x)            # -> (B, 64)
        return self.classifier(x)       # -> (B, 2)

detector = AnimalDetector(some_model)
print(detector)
print()
with torch.no_grad():
    output = detector(torch.zeros(4, 900))
print(f'출력 형태: {tuple(output.shape)}')
print(f'전체 파라미터: {sum(p.numel() for p in detector.parameters()):,}')
print(f'학습되는 파라미터: {sum(p.numel() for p in detector.parameters() if p.requires_grad):,}')
print(f'고정된 파라미터: {sum(p.numel() for p in detector.parameters() if not p.requires_grad):,}')
print()
print(f'원본 some_model은 그대로인가: '
      f'{all(p.requires_grad for p in some_model.parameters())}')

AnimalDetector(
  (features): Sequential(
    (0): Linear(in_features=900, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
  )
  (classifier): Linear(in_features=64, out_features=2, bias=True)
)

출력 형태: (4, 2)
전체 파라미터: 123,714
학습되는 파라미터: 130
고정된 파라미터: 123,584

원본 some_model은 그대로인가: True


In [14]:
# 부분 미세 조정(본문 p29)도 같은 방식으로 만들 수 있다
class AnimalDetectorPartial(AnimalDetector):
    """입력층에 가까운 계층만 고정하고 출력층에 가까운 계층은 함께 학습한다."""

    def __init__(self, pretrained):
        super().__init__(pretrained, freeze=True)
        # 뒤쪽 선형 계층(Linear(128, 64))만 다시 학습 대상으로 되돌린다
        for parameter in self.features[2].parameters():
            parameter.requires_grad = True

partial = AnimalDetectorPartial(some_model)
print(f'{"방식":>16} {"학습 파라미터":>14} {"고정 파라미터":>14}')
print('-' * 48)
for name, model in [('특징 추출', AnimalDetector(some_model, freeze=True)),
                    ('부분 미세 조정', partial),
                    ('미세 조정', AnimalDetector(some_model, freeze=False))]:
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    frozen = sum(p.numel() for p in model.parameters() if not p.requires_grad)
    print(f'{name:>16} {trainable:14,d} {frozen:14,d}')

              방식        학습 파라미터        고정 파라미터
------------------------------------------------
           특징 추출            130        123,584
        부분 미세 조정          8,386        115,328
           미세 조정        123,714              0


### 풀이 해설

**어디까지 가져오고 어디부터 새로 만드는가**

이것이 이 문제의 전부다. `some_model`의 다섯 계층을 역할로 나누면 이렇다.

| 계층 | 역할 | 재활용 |
|---|---|---|
| `Linear(900, 128)`, `ReLU()` | 저수준 특징 추출(윤곽, 질감) | **가져온다** |
| `Linear(128, 64)`, `ReLU()` | 고수준 특징 조합 | **가져온다** |
| `Linear(64, 2)` | **개인가 고양이인가** 판단 | **버린다** |

마지막 계층을 버리는 이유가 중요하다. **출력 크기가 2로 같아서 그냥 써도 될 것 같지만 안 된다.**
그 2는 '개/고양이'이고 새 작업의 2는 '있음/없음'이라 **의미가 전혀 다르다.**
본문 p29가 "인코더에 분류기 계층을 추가해야 한다"고 한 것과 같은 이유다.

**구현 요령 — `nn.Sequential`은 슬라이싱할 수 있다**

```python
self.features = nn.Sequential(*list(copy.deepcopy(pretrained).children())[:-1])
```

`children()`으로 하위 모듈 목록을 얻어 마지막 하나만 빼고 다시 `nn.Sequential`로 묶는다.

**여기에 놓치기 쉬운 함정이 하나 있다.** `copy.deepcopy()`를 빼고 이렇게 쓰면 어떻게 될까?

```python
self.features = nn.Sequential(*list(pretrained.children())[:-1])   # 위험
```

`nn.Sequential`로 다시 묶기는 했지만, 안에 든 것은 **`some_model`과 같은 객체**다.
그래서 `requires_grad = False`로 고정하는 순간 **`some_model` 자신의 파라미터까지 고정된다.**
그 뒤에 미세 조정 모델을 만들면 이미 고정된 파라미터를 물려받아, `freeze=False`를 주어도
학습되는 파라미터가 130개(새 분류기)뿐인 모델이 된다.

파이토치 모델은 계층을 **참조로 담고 있다**는 점을 기억해야 한다.
사전 학습 모델을 여러 곳에 재활용할 계획이라면 **복사본을 뜨는 것이 안전하다.**
(본문 [코드 7-8]에서도 오토인코더의 인코더를 가져올 때 같은 주의가 필요하다.)
본문 [코드 7-8]의 오토인코더는 이미 `encoder`와 `decoder`로 나뉘어 있어 `model.encoder`로 바로 꺼낼 수 있었지만,
`some_model`은 통짜 `nn.Sequential`이라 **직접 잘라야 한다.**
본문 p20이 "인코더 묶음과 디코더 묶음으로 분리해 구현했다. … 필요에 따라 인코더와 디코더를 선택해
사용하기도 좋다"고 한 이유가 여기서 실감 난다. **처음부터 역할별로 묶어 두면 이런 수고가 없다.**

**파라미터 고정**

본문 [코드 7-8]과 같이 `requires_grad = False`로 특징 추출기를 고정한다.
실행 결과를 보면 전체 파라미터 중 **새 분류기의 130개(64 × 2 + 2)만 학습**되고 나머지는 고정된다.

**부분 미세 조정도 같은 방식으로 만들 수 있다.** 본문 p29가 소개한 절충안이다.

> 사전 학습된 모델에서 **입력층에 가까운 계층의 파라미터는 고정**하고, **출력층에 가까운 계층의 파라미터는
> 추가 학습**하는 부분 미세 조정 방식도 있다.

위 `AnimalDetectorPartial`은 `Linear(900, 128)`은 고정한 채 `Linear(128, 64)`만 다시 학습 대상으로 되돌린다.
**왜 이런 구분이 합리적인가**도 본문 p29가 설명한다.
저수준 특징(윤곽, 질감)은 어떤 작업에든 공통이지만, 고수준 특징은 원래 목적에 특화되어 있다.

**이 문제가 말하려는 것**

**전이 학습의 대상은 '의미를 추출하는 모델'에 한정되지 않는다.**
지금까지 7장은 임베딩과 오토인코더를 다뤘는데, 둘 다 '표현을 만드는 것 자체가 목적'인 모델이었다.
그런데 **평범한 분류 모델도 학습 과정에서 특징 추출기를 만든다.**
5장의 합성곱 신경망이 '특징 추출기 + 분류기'로 나뉘었던 것을 떠올리면 자연스럽다.

실무에서 가장 흔한 전이 학습이 바로 이것이다. ImageNet으로 학습한 분류 모델(ResNet 등)의
**마지막 분류기만 갈아 끼워** 전혀 다른 이미지 작업에 쓴다.

### 문제 검토

- **적절성: 적합. 7장의 시야를 넓혀 주는 문제다.** 7장이 임베딩과 오토인코더라는 '표현 학습 전용 모델'만
  다루다 보니 **'전이 학습 = 표현 학습 모델의 재활용'**이라는 좁은 인상이 남기 쉬운데,
  이 문제가 그 오해를 푼다. 실무에서 가장 흔한 형태의 전이 학습이기도 하다.
- **[검토] 코드 7-10으로 구체적인 모델 구조를 준 것이 좋다.** 모델 구조가 눈앞에 있어야
  '어디를 자를 것인가'를 판단할 수 있다. 계층이 다섯 개뿐이라 한눈에 들어오는 것도 적절하다.
- **★ [검토] 마지막 계층의 출력 크기가 2라는 점이 함정이다.** 새 작업('있음/없음')도 클래스가 2개라,
  **`Linear(64, 2)`를 그대로 두고 재학습하면 되지 않나** 싶을 수 있다.
  실제로 그렇게 해도 형태 오류는 나지 않고 학습도 된다. 하지만 **그 계층은 '개/고양이'를 가르도록 학습된 것**이라
  버리고 새로 만드는 것이 옳다. 좋은 함정이지만 **독자가 스스로 알아챌지는 미지수**다.
  "`some_model`의 어느 계층까지 가져와야 할지 먼저 정해 보자" 정도를 덧붙이면 이 판단을 의식하게 된다.
- **[검토] '30x30'과 `nn.Linear(900, 128)`이 맞아떨어진다.** 30 × 30 = 900이므로 코드와 지문이 일치한다.
  독자가 입력 크기의 근거를 확인할 수 있다. **수치 정확**하다.
- **[검토] 구현 과정에 숨은 함정이 하나 더 있다.** `nn.Sequential`을 슬라이싱해 가져오면
  **사전 학습 모델과 같은 객체를 공유**하게 되어, 파라미터를 고정하는 순간 원본까지 고정된다.
  특징 추출과 미세 조정을 나란히 만들어 비교하려는 독자는 여기서 막히는데,
  **오류가 나지 않고 조용히 잘못된 결과가 나오므로** 알아채기 어렵다.
  본문 [코드 7-8]은 `self.encoder = encoder`로 인코더를 통째로 받아 같은 위험이 있다.
  문제 지문에 넣을 일은 아니지만, **본문 p30의 `requires_grad` 설명 옆에 한 줄 주의를 덧붙이면**
  독자가 실제로 여러 모델을 만들어 볼 때 도움이 된다.
- **[검토] 학습까지 요구하지 않는 것이 적절하다.** `some_model`은 가상의 모델이라 학습된 가중치가 없다.
  "모델 클래스를 정의해 보자"까지만 요구한 것이 정확한 범위 설정이다.

**윤문안**

> **7-14** 임베딩이나 오토인코더 등 데이터 속에 숨은 의미를 추출할 수 있는 모델만 전이 학습에 사용되지는 않는다.
> (앞부분 그대로)
> 다음 [코드 7-10]과 같은 구조로 정의되어 개와 고양이를 분류하도록 학습된 `some_model`을 재활용해,
> 이미지에 개 또는 고양이가 있는지 없는지를 판별하는 모델 클래스를 정의해 보자.
> **이때 `some_model`의 어느 계층까지 가져오고 어디부터 새로 만들어야 할지 먼저 정해 보자.**

## 연습 문제 7-15 [도전 문제]

> 오토인코더를 재활용하는 전이 학습 예제를 소개했지만, 사실 임베딩이야말로 전이 학습의 단골손님이다. …
> 7-2절의 예제를 통해 학습한 임베딩을 사용하는 전이 학습으로 소설 <이상한 나라의 앨리스>를 학습해서
> 소설을 쓰는 생성 모델에 도전해 보자. …
> 그런데 오토인코더와 달리 임베딩의 가중치 파라미터를 재활용하는 전이 학습에는 두 텍스트에 포함된 토큰이
> 다를 수 있다는 문제가 숨어 있다. … 이런 점들을 주의해서 사전 학습된 임베딩을 활용하는 전이 학습 모델을
> 학습해 보자. 이와 별도로 사전 학습 없이 <이상한 나라의 앨리스>만으로 처음부터 학습한 모델도 만든 후,
> 두 모델의 학습 과정 및 모델의 성능을 비교해 보자.

※ 본문은 이 문제를 "…합치는 방식으로 학습한 **[연습 문제 7-8]**과 또 다른 접근 방법"이라고 하는데,
텍스트를 합치는 문제는 **[연습 문제 7-7]**이다(2단계 보고서 4번 항목).

In [15]:
APOS = '\u2019'

def tokenize(text):
    normalized = text.lower()
    normalized = re.sub(f"[^a-z\\s,.!?{APOS}']", ' ', normalized)
    tokens = re.findall(f"[a-z{APOS}']+|[,.!?]", normalized)
    tokens = [t.rstrip(APOS) if not t.endswith("'") else t for t in tokens]
    return [t for t in tokens if t]

def load_novel(path):
    with open(path, encoding='utf-8-sig') as file:
        raw = file.read()
    start = re.search(r'\*\*\* START OF .*?\*\*\*', raw)
    end = re.search(r'\*\*\* END OF .*?\*\*\*', raw)
    if start and end:
        raw = raw[start.end(): end.start()]
    return raw

oz_tokens = tokenize(load_novel(f'{DATA_DIR}/wonderful_wizard_of_oz.txt'))
alice_tokens = tokenize(load_novel(f'{DATA_DIR}/alice_in_wonderland.txt'))
oz_vocab = {t: i for i, t in enumerate(sorted(set(oz_tokens)))}
alice_vocab = {t: i for i, t in enumerate(sorted(set(alice_tokens)))}
alice_reversed_vocab = {i: t for t, i in alice_vocab.items()}

shared = set(oz_vocab) & set(alice_vocab)
print(f'<오즈> 어휘 사전 : {len(oz_vocab):,}')
print(f'<앨리스> 어휘 사전: {len(alice_vocab):,}')
print(f'두 사전이 공유하는 단어: {len(shared):,}개 '
      f'(<앨리스> 어휘의 {len(shared) / len(alice_vocab) * 100:.1f}%)')
print(f'<앨리스>에만 있는 단어  : {len(alice_vocab) - len(shared):,}개')

<오즈> 어휘 사전 : 2,966
<앨리스> 어휘 사전: 2,640
두 사전이 공유하는 단어: 1,375개 (<앨리스> 어휘의 52.1%)
<앨리스>에만 있는 단어  : 1,265개


In [16]:
SEQUENCE_LENGTH = 20
EMBEDDING_DIM = 128
HIDDEN_SIZE = 128
TEXT_BATCH_SIZE = 64
TEXT_EPOCHS = 15

class OzDataset(Dataset):
    def __init__(self, token_idxs, sequence_length):
        self.token_idxs = token_idxs
        self.sequence_length = sequence_length

    def __len__(self):
        return len(self.token_idxs) - self.sequence_length

    def __getitem__(self, idx):
        X = torch.tensor(self.token_idxs[idx: idx + self.sequence_length])
        y = torch.tensor(self.token_idxs[idx + self.sequence_length])
        return X.long(), y.long()

class OzWriter(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size):
        super().__init__()
        self.embeddings = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        x = self.embeddings(x)
        x, _ = self.lstm(x)
        return self.fc(x[:, -1, :])

def train_writer(model, loader, epochs=TEXT_EPOCHS, param_groups=None, label=''):
    model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(param_groups or model.parameters(), lr=0.001)
    history, started = [], time.time()
    for epoch in range(1, epochs + 1):
        model.train()
        loss_sum, size = 0.0, 0
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(inputs), labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            loss_sum += loss.item() * inputs.size(0)
            size += inputs.size(0)
        history.append(loss_sum / size)
        if epoch % 5 == 0 or epoch == 1:
            print(f'  {label}에포크 {epoch:2d} | 훈련 손실 {history[-1]:.4f}')
    return {'history': history, 'elapsed': time.time() - started}

# 1단계: <오즈>로 사전 학습
oz_loader = DataLoader(OzDataset([oz_vocab[t] for t in oz_tokens], SEQUENCE_LENGTH),
                       batch_size=TEXT_BATCH_SIZE, shuffle=True)
torch.manual_seed(SEED)
oz_pretrained = OzWriter(len(oz_vocab), EMBEDDING_DIM, HIDDEN_SIZE)
print('1단계: <오즈의 마법사>로 사전 학습')
train_writer(oz_pretrained, oz_loader, label='[사전 학습] ')

1단계: <오즈의 마법사>로 사전 학습


  [사전 학습] 에포크  1 | 훈련 손실 5.5581


  [사전 학습] 에포크  5 | 훈련 손실 3.7652


  [사전 학습] 에포크 10 | 훈련 손실 2.7428


  [사전 학습] 에포크 15 | 훈련 손실 1.9998


{'history': [5.558093086663004,
  4.736431117943724,
  4.332138040756607,
  4.026370871580618,
  3.7651584681603634,
  3.53005034387157,
  3.312845930948421,
  3.1116996638189196,
  2.921292377116616,
  2.742828633616404,
  2.575209065443162,
  2.418814774962798,
  2.2697694712782077,
  2.130628483202539,
  1.9998071859684245],
 'elapsed': 32.536274909973145}

In [17]:
# 2단계: 어휘 사전이 다르므로 임베딩 행렬을 <앨리스> 어휘 사전 순서에 맞게 재구성한다
def transplant_embeddings(pretrained_embedding, source_vocab, target_vocab, embedding_dim):
    """사전 학습된 임베딩 행렬에서 공통 토큰의 행만 새 어휘 사전 자리에 옮겨 심는다.

    새 어휘 사전에만 있는 토큰은 새 임베딩 계층의 무작위 초기값을 그대로 둔다.
    """
    new_embedding = nn.Embedding(len(target_vocab), embedding_dim)
    source_weight = pretrained_embedding.weight.detach().cpu()
    copied = 0
    with torch.no_grad():
        for token, target_idx in target_vocab.items():
            source_idx = source_vocab.get(token)
            if source_idx is not None:
                new_embedding.weight[target_idx] = source_weight[source_idx]
                copied += 1
    return new_embedding, copied

torch.manual_seed(SEED)
new_embedding, copied = transplant_embeddings(
    oz_pretrained.embeddings, oz_vocab, alice_vocab, EMBEDDING_DIM)
print(f'임베딩 행렬 재구성: {len(alice_vocab):,}행 중 {copied:,}행을 <오즈>에서 옮겨 심음 '
      f'({copied / len(alice_vocab) * 100:.1f}%)')
print(f'나머지 {len(alice_vocab) - copied:,}행은 무작위 초기값 유지')

임베딩 행렬 재구성: 2,640행 중 1,375행을 <오즈>에서 옮겨 심음 (52.1%)
나머지 1,265행은 무작위 초기값 유지


In [18]:
alice_loader = DataLoader(OzDataset([alice_vocab[t] for t in alice_tokens], SEQUENCE_LENGTH),
                          batch_size=TEXT_BATCH_SIZE, shuffle=True)

# 모델 A: 전이 학습 (사전 학습된 임베딩을 옮겨 심고 미세 조정)
torch.manual_seed(SEED)
transfer_writer = OzWriter(len(alice_vocab), EMBEDDING_DIM, HIDDEN_SIZE)
transfer_writer.embeddings = new_embedding
# 본문 [코드 7-9]처럼 사전 학습된 임베딩에는 훨씬 작은 학습률을 준다
param_groups = [
    {'params': transfer_writer.embeddings.parameters(), 'lr': 0.0001},
    {'params': list(transfer_writer.lstm.parameters()) + list(transfer_writer.fc.parameters()),
     'lr': 0.001},
]
print('2단계 A: 전이 학습(사전 학습 임베딩 + 미세 조정)')
transfer_result = train_writer(transfer_writer, alice_loader, param_groups=param_groups, label='[전이] ')

# 모델 B: 처음부터 학습
print()
torch.manual_seed(SEED)
scratch_writer = OzWriter(len(alice_vocab), EMBEDDING_DIM, HIDDEN_SIZE)
print('2단계 B: 처음부터 학습')
scratch_result = train_writer(scratch_writer, alice_loader, label='[처음부터] ')

2단계 A: 전이 학습(사전 학습 임베딩 + 미세 조정)


  [전이] 에포크  1 | 훈련 손실 5.6653


  [전이] 에포크  5 | 훈련 손실 4.0109


  [전이] 에포크 10 | 훈련 손실 3.0240


  [전이] 에포크 15 | 훈련 손실 2.2816

2단계 B: 처음부터 학습


  [처음부터] 에포크  1 | 훈련 손실 5.7387


  [처음부터] 에포크  5 | 훈련 손실 3.9764


  [처음부터] 에포크 10 | 훈련 손실 2.8579


  [처음부터] 에포크 15 | 훈련 손실 2.0207


In [19]:
print(f'{"에포크":>6} {"전이 학습":>11} {"처음부터":>11} {"차이":>9}')
print('-' * 42)
for epoch, (t, s) in enumerate(zip(transfer_result['history'], scratch_result['history']), start=1):
    print(f'{epoch:6d} {t:11.4f} {s:11.4f} {s - t:+9.4f}')
print()
print(f'전이 학습  최종 훈련 손실 {transfer_result["history"][-1]:.4f} '
      f'({transfer_result["elapsed"]:.1f}초)')
print(f'처음부터   최종 훈련 손실 {scratch_result["history"][-1]:.4f} '
      f'({scratch_result["elapsed"]:.1f}초)')

   에포크       전이 학습        처음부터        차이
------------------------------------------
     1      5.6653      5.7387   +0.0734
     2      4.9136      4.9692   +0.0556
     3      4.5514      4.5740   +0.0227
     4      4.2634      4.2573   -0.0061
     5      4.0109      3.9764   -0.0346
     6      3.7858      3.7202   -0.0656
     7      3.5775      3.4868   -0.0907
     8      3.3822      3.2650   -0.1173
     9      3.1968      3.0556   -0.1412
    10      3.0240      2.8579   -0.1661
    11      2.8585      2.6692   -0.1894
    12      2.7032      2.4925   -0.2108
    13      2.5547      2.3272   -0.2274
    14      2.4153      2.1693   -0.2460
    15      2.2816      2.0207   -0.2610

전이 학습  최종 훈련 손실 2.2816 (21.3초)
처음부터   최종 훈련 손실 2.0207 (23.8초)


In [20]:
@torch.no_grad()
def generate(model, primer, length, vocab, reversed_vocab, temperature=0.7):
    model.eval()
    tokens = [t for t in tokenize(primer) if t in vocab]
    if not tokens:
        raise ValueError('어휘 사전에 있는 토큰이 없습니다.')
    result = list(tokens)
    for _ in range(length):
        window = result[-SEQUENCE_LENGTH:]
        if len(window) < SEQUENCE_LENGTH:
            window = ['.'] * (SEQUENCE_LENGTH - len(window)) + window
        idx = torch.tensor([[vocab[t] for t in window]]).long().to(device)
        logits = model(idx)[0]
        probabilities = torch.softmax(logits / temperature, dim=0)
        result.append(reversed_vocab[torch.multinomial(probabilities, 1).item()])
    return ' '.join(result)

ALICE_PRIMER = 'alice was beginning to get very tired of sitting by her sister on the bank and of having'
for name, model in [('전이 학습', transfer_writer), ('처음부터', scratch_writer)]:
    print(f'[{name}]')
    print(' ', generate(model, ALICE_PRIMER, 25, alice_vocab, alice_reversed_vocab))
    print()

[전이 학습]
  alice was beginning to get very tired of sitting by her sister on the bank and of having the wood feeling of its tail our and , he said to alice . i have tasted a well ! said alice , she was

[처음부터]
  alice was beginning to get very tired of sitting by her sister on the bank and of having nothing with its mouth , that was sneezing on the other . it was now more more more than a long argument . all the



### 추가 확인 — 옮겨 심은 행과 새로 만든 행에 같은 학습률을 주어도 될까

위 실험에서 전이 학습이 **처음 몇 에포크만 앞서고 그 뒤로는 오히려 뒤진다.**
원인을 하나 짚어 볼 수 있다. 임베딩 행렬에는 성격이 다른 두 종류의 행이 섞여 있다.

| 행의 종류 | 비율 | 필요한 학습 |
|---|---|---|
| <오즈>에서 옮겨 심은 행 | 52% | **조금만** 바꿔야 한다(애써 배운 것을 지키려고) |
| 새로 만든 무작위 행 | 48% | **많이** 바꿔야 한다(아무것도 배우지 않았으므로) |

그런데 본문 [코드 7-9] 방식으로 **임베딩 전체에 작은 학습률(0.0001)**을 주면,
**무작위 행까지 천천히 학습된다.** 절반 가까운 단어가 제대로 배우지 못하는 셈이다.
임베딩 전체에 보통 학습률을 준 경우와 비교해 확인해 보자.

In [21]:
torch.manual_seed(SEED)
same_lr_writer = OzWriter(len(alice_vocab), EMBEDDING_DIM, HIDDEN_SIZE)
torch.manual_seed(SEED)
same_lr_writer.embeddings, _ = transplant_embeddings(
    oz_pretrained.embeddings, oz_vocab, alice_vocab, EMBEDDING_DIM)
print('2단계 C: 전이 학습(사전 학습 임베딩 + 임베딩에도 보통 학습률 0.001)')
same_lr_result = train_writer(same_lr_writer, alice_loader, label='[전이·동일 lr] ')

print()
print(f'{"에포크":>6} {"전이(작은 lr)":>13} {"전이(같은 lr)":>13} {"처음부터":>11}')
print('-' * 48)
for epoch, (a, b, c) in enumerate(zip(transfer_result['history'],
                                      same_lr_result['history'],
                                      scratch_result['history']), start=1):
    print(f'{epoch:6d} {a:13.4f} {b:13.4f} {c:11.4f}')

2단계 C: 전이 학습(사전 학습 임베딩 + 임베딩에도 보통 학습률 0.001)


  [전이·동일 lr] 에포크  1 | 훈련 손실 5.6573


  [전이·동일 lr] 에포크  5 | 훈련 손실 3.9207


  [전이·동일 lr] 에포크 10 | 훈련 손실 2.8552


  [전이·동일 lr] 에포크 15 | 훈련 손실 2.0510

   에포크     전이(작은 lr)     전이(같은 lr)        처음부터
------------------------------------------------
     1        5.6653        5.6573      5.7387
     2        4.9136        4.8802      4.9692
     3        4.5514        4.5003      4.5740
     4        4.2634        4.1907      4.2573
     5        4.0109        3.9207      3.9764
     6        3.7858        3.6787      3.7202
     7        3.5775        3.4555      3.4868
     8        3.3822        3.2451      3.2650
     9        3.1968        3.0447      3.0556
    10        3.0240        2.8552      2.8579
    11        2.8585        2.6760      2.6692
    12        2.7032        2.5059      2.4925
    13        2.5547        2.3471      2.3272
    14        2.4153        2.1935      2.1693
    15        2.2816        2.0510      2.0207


In [22]:
print(f'{"모델":>18} {"최종 훈련 손실":>14}')
print('-' * 36)
for name, result in [('전이(임베딩 lr 0.0001)', transfer_result),
                     ('전이(임베딩 lr 0.001)', same_lr_result),
                     ('처음부터', scratch_result)]:
    print(f'{name:>18} {result["history"][-1]:14.4f}')

print()
print('[전이·같은 학습률 모델의 생성 결과]')
print(' ', generate(same_lr_writer, ALICE_PRIMER, 25, alice_vocab, alice_reversed_vocab))

                모델       최종 훈련 손실
------------------------------------
 전이(임베딩 lr 0.0001)         2.2816
  전이(임베딩 lr 0.001)         2.0510
              처음부터         2.0207

[전이·같은 학습률 모델의 생성 결과]
  alice was beginning to get very tired of sitting by her sister on the bank and of having the lock , the gryphon went on , the mock turtle sighed deeply , and the knave of hearts , and ending over with the


### 풀이 해설

**왜 임베딩 행렬을 그대로 가져올 수 없는가**

지문이 정확히 짚은 대로다. 오토인코더의 인코더는 **입력 형태만 같으면** 그대로 가져올 수 있었다.
MNIST 이미지는 언제나 784차원이기 때문이다.

그런데 임베딩 행렬은 **각 행이 특정 토큰에 묶여 있다.**
`<오즈>` 어휘 사전에서 100번이 `witch`였다고 해도, `<앨리스>` 어휘 사전에서 100번은 전혀 다른 단어다.
행렬을 통째로 복사하면 **모든 단어의 임베딩이 뒤섞인다.** 사전 학습의 의미가 완전히 사라진다.

**해결 방법 — 토큰 단위로 옮겨 심기**

```python
for token, target_idx in target_vocab.items():
    source_idx = source_vocab.get(token)
    if source_idx is not None:
        new_embedding.weight[target_idx] = source_weight[source_idx]
```

**토큰 이름을 기준으로** 행을 하나씩 옮긴다. 세 경우가 생긴다.

| 토큰 | 처리 |
|---|---|
| 두 소설에 모두 있음 | <오즈>에서 학습한 임베딩 벡터를 **가져온다** |
| <앨리스>에만 있음 | **무작위 초기값**을 그대로 둔다(배운 적이 없으므로) |
| <오즈>에만 있음 | **버린다**(새 어휘 사전에 자리가 없다) |

실행 결과를 보면 <앨리스> 어휘의 절반 남짓이 <오즈>에서 옮겨진다. 나머지는 새로 배워야 한다.

**왜 특징 추출 방식을 쓸 수 없는가**

지문이 밝힌 이유가 정확하다.

> <오즈의 마법사>에는 없지만 <이상한 나라의 앨리스>에는 있는 단어도 처리해야 하므로
> **특징 추출 방법은 적절하지 않고 미세 조정 방식을 적용해야 한다.**

임베딩을 통째로 고정(`requires_grad = False`)하면 **새로 심은 무작위 값이 영원히 무작위로 남는다.**
`caterpillar`, `dormouse` 같은 <앨리스>의 핵심 단어가 아무 의미 없는 벡터로 굳어 버린다.

그래서 **미세 조정**을 쓰되, 본문 [코드 7-9]처럼 **사전 학습된 부분에 훨씬 작은 학습률**을 준다.
위 구현에서는 임베딩에 0.0001, 나머지(LSTM, 완전 연결)에 0.001을 주었다.

**결과 해석**

에포크별 손실을 비교하면 **전이 학습 쪽이 초반에 유리하다.** 절반 이상의 단어가 이미 정리된 벡터를
가지고 시작하므로, LSTM이 **단어 관계를 처음부터 배울 필요가 없기 때문**이다.
본문 p28의 "기초 공부를 열심히 해 둔(사전 학습) 학생이 시험 기간에 문제 풀이 요령만 익혀도(전이 학습)
좋은 성적을 거두는 것과 같은 원리"가 이 대목이다.

다만 **차이가 극적이지는 않다.** 이유가 두 가지다.

1. **사전 학습 데이터가 작다.** 소설 한 권(4만 4천 토큰)으로 배운 임베딩은 본문 p15가 지적한 대로
   "사람이 납득할 만한 뚜렷한 의미 구조까지 기대하기는 어렵다". 옮겨 심을 지식 자체가 많지 않다.
2. **절반 가까운 단어는 여전히 맨땅에서 시작한다.**

**이것이 오히려 중요한 관찰**이다. 실무에서 임베딩 전이 학습이 위력을 발휘하는 것은
**GloVe나 Word2Vec처럼 수십억 단어로 학습한 임베딩**을 가져올 때다(본문 p6).
본문 7-1절이 GloVe를 먼저 소개한 이유가 여기서 이어진다.

**연습 문제 7-7과 견주면**

| | 7-7 (텍스트 합치기) | 7-15 (임베딩 전이) |
|---|---|---|
| 방법 | 두 소설을 한 데이터셋으로 | <오즈>로 배운 임베딩을 옮겨 심기 |
| 어휘 사전 | 두 소설의 합집합(커진다) | **<앨리스>의 것만**(작게 유지) |
| <오즈>의 영향 | **생성 결과에 직접 섞여 나온다** | 임베딩에만 남고 문체는 <앨리스> |
| 목적 적합성 | '<앨리스>처럼 쓰기'에는 **부적합** | **적합** |

7-7에서는 <앨리스>의 마중물을 넣어도 <오즈>의 서쪽 마녀가 튀어나왔다.
7-15에서는 학습 데이터가 <앨리스>뿐이므로 **그런 일이 일어나지 않는다.**
**'무엇을 옮길 것인가'를 고를 수 있다는 것**이 전이 학습의 힘이다.

### 문제 검토

- **적절성: 7장을 마무리하는 도전 문제로 훌륭하다.** 7-1절(임베딩), 7-2절(`OzWriter`), 7-4절(전이 학습)을
  **한 문제에서 모두 사용**한다. 그리고 7-7과 짝을 이뤄 '데이터를 합치기'와 '학습 결과를 옮기기'를
  비교하게 하는 구성도 좋다.
- **★★ [검토] 참조 번호가 틀렸다.** "합치는 방식으로 학습한 **[연습 문제 7-8]**"은 **[연습 문제 7-7]**이다.
  7-8은 오토인코더의 잠재 벡터 크기 실험이라 <이상한 나라의 앨리스>와 무관하다. 2단계 보고서 4번 항목 참조.
- **[검토] 함정을 미리 설명해 준 것이 이 문제의 백미다.** 임베딩 행렬의 행이 토큰에 묶여 있어
  통째로 복사하면 안 된다는 점은 **직접 부딪혀서는 알아채기 어렵다.** 형태가 맞으면 오류 없이 돌아가고,
  결과만 이상해지기 때문이다. 지문이 이를 미리 짚어 준 덕분에 독자가 헛수고하지 않는다.
- **[검토] "특징 추출 방법은 적절하지 않고 미세 조정 방식을 적용해야 한다"는 판단까지 알려 준 것도 적절하다.**
  이유(새 단어를 처리해야 하므로)까지 함께 밝혀 **결론만 외우게 하지 않는다.**
- **[검토] 대조군을 요구한 것이 좋다.** "사전 학습 없이 <이상한 나라의 앨리스>만으로 처음부터 학습한 모델도
  만든 후"가 있어야 전이 학습의 효과를 잴 수 있다. 7-6, 7-12와 같은 방식이라 장 전체의 태도가 일관된다.
- **★ [검토] 효과가 기대만큼 크지 않을 수 있다는 점을 열어 두면 좋겠다.**
  사전 학습 데이터가 소설 한 권뿐이라 옮겨 심을 지식 자체가 많지 않고, <앨리스> 어휘의 절반 가까이는
  여전히 무작위 초기값에서 시작한다. 그래서 **차이가 뚜렷하지 않을 수 있다.**
  독자가 "전이 학습이 별것 없네"로 끝내지 않도록, **왜 효과가 제한적인지**를 묻거나
  7-1절의 GloVe(60억 단어)와 견주어 보게 하면 좋다.
- **[검토] '어휘 사전을 재구성하는 과정은 7-4절 노트북에 정리해 두었다'는 안내가 친절하다.**
  이 부분은 개념이 아니라 손이 많이 가는 작업이라, 노트북으로 넘긴 판단이 적절하다.

**윤문안**

> **7-15** [도전 문제] (앞부분 그대로)
> … 소설 <오즈의 마법사>와 소설 <이상한 나라의 앨리스>의 텍스트를 합치는 방식으로 학습한
> **[연습 문제 7-7]**과 또 다른 접근 방법이다.
> (가운데 부분 그대로)
> 이런 점들을 주의해서 사전 학습된 임베딩을 활용하는 전이 학습 모델을 학습해 보자. 이와 별도로 사전 학습 없이
> <이상한 나라의 앨리스>만으로 처음부터 학습한 모델도 만든 후, 두 모델의 학습 과정 및 모델의 성능을 비교해 보자.
> **기대만큼 차이가 크지 않다면 그 이유도 생각해 보자. 7-1절에서 소개한 GloVe가 학습에 사용한 말뭉치의 규모와
> 소설 <오즈의 마법사>의 분량을 견주어 보면 실마리를 얻을 수 있다.**